# Модель кредитного скоринга

**Цель:** прогноз дефолта предприятий малого бизнеса на горизонте 12 месяцев (`default_12m`).

## Данные

Использовано 20 000 синтетических кредитных заявок. Датасет включает 15 базовых признаков и 4 расчётные финансовые метрики (Feature Engineering).

Целевая доля дефолтов откалибрована примерно на 7%.

## Признаки модели

| Поле | Описание |
|---|---|
| `amount` | Сумма кредита |
| `term_months` | Срок кредита в месяцах |
| `purpose` | Цель кредита |
| `legal_form` | Организационно-правовая форма (ИП/ООО) |
| `has_collateral` | Наличие залога |
| `collateral_type` | Тип залога |
| `business_age_months` | Возраст бизнеса в месяцах |
| `annual_revenue` | Годовая выручка |
| `monthly_turnover` | Ежемесячный оборот |
| `monthly_expenses` | Ежемесячные расходы |
| `existing_debt` | Текущая задолженность |
| `monthly_debt_payments` | Ежемесячные платежи по долгам |
| `overdue_days` | Количество дней просрочки |
| `collateral_value` | Стоимость залога |
| `industry` | Отрасль бизнеса |
| `expense_ratio` | Доля расходов в обороте (расчётный признак) |
| `debt_ratio` | Доля платежей по долгам в обороте (расчётный признак) |
| `loan_ratio` | Отношение суммы кредита к годовой выручке (расчётный признак) |
| `collateral_coverage` | Коэффициент покрытия кредита залогом (расчётный признак) |

**Целевая переменная:** `default_12m` — факт дефолта в течение 12 месяцев (`1` — дефолт, `0` — нет).

## Генерация и подготовка данных

Сгенерировано 20 000 синтетических заявок с учётом финансовых показателей, долговой нагрузки, просрочек и стоимости залога. В алгоритме генерации предусмотрены нелинейные зависимости и взаимодействия признаков, например повышенный риск для молодых компаний в строительной отрасли.

Вероятности дефолта (PD) настроены на целевую среднюю долю дефолтов около 7%. Данные разделены на обучающую и тестовую выборки в пропорции 80/20 до этапа предобработки.

## Технологии

- **Python** — основной язык разработки.
- **Pandas, NumPy** — генерация данных, Feature Engineering и финансовое моделирование.
- **Scikit-learn** — построение пайплайнов (`ColumnTransformer`, `Pipeline`), стандартизация, кодирование категорий, оценка порогов и обучение моделей.
- **CatBoost** — модель градиентного бустинга с поддержкой категориальных признаков.
- **LightGBM** — дополнительный алгоритм для сравнения качества.

## Сравнение алгоритмов

Для оценки качества моделей при дисбалансе классов использовались **PR-AUC (Average Precision)** и **ROC-AUC**.

Сравнивались Logistic Regression, Random Forest, CatBoost и LightGBM. В описанном эксперименте модели обучались без искусственной балансировки классов.

**Лучшая модель по результатам сравнения — CatBoost.**

### Результаты CatBoost на тестовой выборке

| Метрика | Значение |
|---|---:|
| ROC-AUC | ~0.789 |
| PR-AUC (Average Precision) | ~0.264 |
| Precision (при пороге, максимизирующем F1) | ~0.341 |
| Recall (при пороге, максимизирующем F1) | ~0.376 |
| Accuracy | ~0.905 |

## Бизнес-оценка и PnL

Для оценки потенциальной экономической эффективности предусмотрена симуляция прибыли и убытков (Profit and Loss, PnL).

Поток заявок распределяется на три зоны:

- **Авто-одобрение** — низкая прогнозная вероятность дефолта.
- **Ручной разбор** — промежуточный уровень риска.
- **Авто-отказ** — высокая прогнозная вероятность дефолта.

Пороги вероятности позволяют настраивать риск-стратегию. Оптимальные значения должны определяться на основании экономических предположений и результатов валидации.

## Артефакты проекта

В результате выполнения кода сохраняются:

- `train.csv` — синтетический датасет с признаками и целевой переменной.
- `risk_model.joblib` — сохранённая модель или пайплайн с обученной моделью, предназначенный для интеграции.

## Ограничения

Модель обучена на синтетических данных. Её качество на реальных заёмщиках не проверялось. Приведённые метрики характеризуют результаты эксперимента на синтетической выборке и не подтверждают точность модели в реальном банковском скоринге.

#Обучение

In [1]:
!pip install -q catboost lightgbm

In [2]:
import glob
import zipfile
import os

zips = glob.glob("/content/*recommende*.zip")
print("Найденные архивы:", zips)

if zips:
    with zipfile.ZipFile(zips[-1]) as z:
        z.extractall("/content/ai-credit-assistant")
    print("Архив распакован")
else:
    print("ZIP не найден — проверь имя файла")

Найденные архивы: []
ZIP не найден — проверь имя файла


In [3]:
!find /content/ai-credit-assistant -name "recommender.py" -o -name "products.json"

find: ‘/content/ai-credit-assistant’: No such file or directory


In [18]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
N = 20_000

PURPOSES = {
    "cash_gap": "кассовый разрыв, налоги, оплата поставщику",
    "inventory": "закупка товара, сезонные закупки",
    "working_capital": "пополнение оборотных средств",
    "equipment": "покупка оборудования",
    "development": "развитие бизнеса, новое направление",
    "refinancing": "рефинансирование кредитов",
    "transport": "покупка транспорта или спецтехники",
    "real_estate": "покупка недвижимости",
}
purposes = list(PURPOSES.keys())

# 1. Генерируем заявки
df = pd.DataFrame({
    "amount": np.clip(rng.lognormal(np.log(2_000_000), 1.0, N), 50_000, 60_000_000).astype(int),
    "term_months": rng.choice([6, 12, 18, 24, 36, 48, 60, 84, 120], N),
    "purpose": rng.choice(purposes, N),
    "legal_form": rng.choice(["IP", "OOO"], N, p=[0.55, 0.45]),
    "has_collateral": (rng.random(N) < 0.4).astype(int),
    "business_age_months": np.clip(rng.gamma(2.0, 20, N).astype(int), 1, 240),
})

df["collateral_type"] = np.where(
    df["has_collateral"].astype(bool),
    rng.choice(["real_estate", "vehicle", "equipment"], N),
    "none",
)

# 2. Финансовые данные
df["annual_revenue"] = rng.lognormal(np.log(15_000_000), 0.8, N).round(0)
df["monthly_turnover"] = (df["annual_revenue"] / 12 * rng.lognormal(0, 0.12, N)).round(0)
df["monthly_expenses"] = (df["monthly_turnover"] * rng.uniform(0.45, 0.92, N)).round(0)
df["existing_debt"] = (df["annual_revenue"] * rng.uniform(0, 0.5, N)).round(0)
df["monthly_debt_payments"] = (df["existing_debt"] * rng.uniform(0.02, 0.10, N)).round(0)
df["overdue_days"] = np.clip(rng.exponential(8, N).astype(int), 0, 120)

df["collateral_value"] = np.where(
    df["has_collateral"].astype(bool),
    (df["amount"] * rng.uniform(0.5, 1.5, N)).round(0),
    0,
)

df["industry"] = rng.choice(
    ["trade", "services", "manufacturing", "construction", "it"],
    N, p=[0.30, 0.30, 0.12, 0.18, 0.10],
)

# 3. Финансовые отношения (сразу пишем в df, чтобы потом отдать моделям)
df["expense_ratio"] = df["monthly_expenses"] / df["monthly_turnover"].clip(lower=1)
df["debt_ratio"] = df["monthly_debt_payments"] / df["monthly_turnover"].clip(lower=1)
df["loan_ratio"] = df["amount"] / df["annual_revenue"].clip(lower=1)
df["collateral_coverage"] = df["collateral_value"] / df["amount"].clip(lower=1)

purpose_risk = df["purpose"].map({
    "cash_gap": 0.15, "inventory": 0.05, "working_capital": 0.00,
    "equipment": -0.15, "development": 0.15, "refinancing": 0.10,
    "transport": 0.00, "real_estate": -0.10,
})

# 4. Формируем риск-скор (добавляем реалистичные нелинейности)
# Штраф: если платежи по долгам съедают больше 40% оборота, риск растет экспоненциально
high_debt_penalty = np.where(df["debt_ratio"] > 0.4, (df["debt_ratio"] - 0.4) * 12, 0)

# Взаимодействие (Interaction feature): молодой бизнес в строительстве = двойной риск
young_construction_risk = ((df["business_age_months"] < 12) & (df["industry"] == "construction")).astype(int) * 0.8

base_logit = (
    1.8 * df["expense_ratio"]
    + 2.0 * df["debt_ratio"] + high_debt_penalty
    + 1.5 * np.log1p(df["loan_ratio"])
    + 0.035 * df["overdue_days"]
    - 0.012 * df["business_age_months"]
    - 0.5 * df["has_collateral"]
    - 0.5 * np.clip(df["collateral_coverage"], 0, 1.5)
    + 0.015 * df["term_months"]
    + purpose_risk
    + young_construction_risk
)

TARGET_DEFAULT_RATE = 0.07

def sigmoid(x):
    x = np.clip(x, -30, 30)
    return 1 / (1 + np.exp(-x))

low, high = -30.0, 10.0
for _ in range(100):
    intercept = (low + high) / 2
    mean_pd = sigmoid(base_logit + intercept).mean()
    if mean_pd < TARGET_DEFAULT_RATE:
        low = intercept
    else:
        high = intercept

intercept = (low + high) / 2
pd_true = sigmoid(base_logit + intercept)

df["default_12m"] = rng.binomial(1, pd_true)

# 5. Формируем датасет (ВКЛЮЧИЛ расчетные фичи)
FEATURES = [
    "amount", "term_months", "purpose", "legal_form", "has_collateral",
    "collateral_type", "business_age_months", "annual_revenue",
    "monthly_turnover", "monthly_expenses", "existing_debt",
    "monthly_debt_payments", "overdue_days", "collateral_value", "industry",
    "expense_ratio", "debt_ratio", "loan_ratio", "collateral_coverage" # Теперь модели их увидят
]

train = df[FEATURES + ["default_12m"]].copy()
train.to_csv("train.csv", index=False)

print("Размер датасета:", train.shape)
print("Свободный член logit:", round(intercept, 3))
print("Фактическая доля дефолтов:", round(train["default_12m"].mean(), 4))

Размер датасета: (20000, 20)
Свободный член logit: -5.247
Фактическая доля дефолтов: 0.0705


In [19]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# 1. Загружаем датасет
df = pd.read_csv("train.csv")

TARGET = "default_12m"

# В список признаков добавлены расчетные финансовые отношения
FEATURES = [
    "amount",
    "term_months",
    "purpose",
    "legal_form",
    "has_collateral",
    "collateral_type",
    "business_age_months",
    "annual_revenue",
    "monthly_turnover",
    "monthly_expenses",
    "existing_debt",
    "monthly_debt_payments",
    "overdue_days",
    "collateral_value",
    "industry",
    "expense_ratio",       # <--- Добавлено
    "debt_ratio",          # <--- Добавлено
    "loan_ratio",          # <--- Добавлено
    "collateral_coverage"  # <--- Добавлено
]

X = df[FEATURES].copy()
y = df[TARGET].astype(int)

# Приводим булево поле к числовому формату
X["has_collateral"] = X["has_collateral"].astype(int)

# 2. Разделяем данные ДО обучения преобразований
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3. Определяем типы признаков
CATEGORICAL = [
    "purpose",
    "legal_form",
    "collateral_type",
    "industry",
]

# Динамически собираем числовые признаки (теперь сюда попадут и новые метрики)
NUMERIC = [
    col for col in FEATURES
    if col not in CATEGORICAL
]

# 4. Обработка числовых признаков:
# пропуски заполняем медианой, затем стандартизируем
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# 5. Обработка категориальных признаков:
# пропуски заполняем самым частым значением,
# категории превращаем в бинарные колонки
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

# 6. Объединяем преобразования
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, NUMERIC),
    ("cat", categorical_pipeline, CATEGORICAL),
])

In [22]:
import joblib
import warnings
from sklearn.metrics import roc_auc_score, average_precision_score

# Гасим технические предупреждения от LightGBM про отсутствие имен колонок
warnings.filterwarnings("ignore", category=UserWarning)

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

# Модели остаются без изменений (без искусственной балансировки)
models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=2000
        ))
    ]),

    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "CatBoost": CatBoostClassifier(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        verbose=False,
        random_seed=42
    ),

    "LightGBM": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=20,
            verbosity=-1,
            random_state=42
        ))
    ])
}

results_roc = {}
results_pr = {}
trained_models = {}

for name, model in models.items():
    if name == "CatBoost":
        model.fit(X_train, y_train, cat_features=[
            "purpose", "legal_form",
            "collateral_type", "industry"
        ])
    else:
        model.fit(X_train, y_train)

    # Получаем честные вероятности дефолта
    pd_pred = model.predict_proba(X_test)[:, 1]

    # Считаем обе метрики
    auc = roc_auc_score(y_test, pd_pred)
    pr_auc = average_precision_score(y_test, pd_pred)

    results_roc[name] = auc
    results_pr[name] = pr_auc
    trained_models[name] = model

    print(f"{name}: ROC-AUC = {auc:.3f} | PR-AUC = {pr_auc:.3f}")

# Сравнение результатов по более строгой метрике
print("\nРезультаты по PR-AUC (важнее для несбалансированных данных):")
for name, pr_auc in sorted(
    results_pr.items(), key=lambda item: item[1], reverse=True
):
    print(f"{name}: {pr_auc:.3f}")

# Выбираем лучшую модель именно по PR-AUC
best_name = max(results_pr, key=results_pr.get)
best_model = trained_models[best_name]

joblib.dump(best_model, "risk_model.joblib")

print(f"\nЛучшая модель: {best_name}")
print("Сохранена в risk_model.joblib")

Logistic Regression: ROC-AUC = 0.785 | PR-AUC = 0.266
Random Forest: ROC-AUC = 0.768 | PR-AUC = 0.288
CatBoost: ROC-AUC = 0.789 | PR-AUC = 0.292
LightGBM: ROC-AUC = 0.782 | PR-AUC = 0.286

Результаты по PR-AUC (важнее для несбалансированных данных):
CatBoost: 0.292
Random Forest: 0.288
LightGBM: 0.286
Logistic Regression: 0.266

Лучшая модель: CatBoost
Сохранена в risk_model.joblib


#Анализ данных

In [25]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    brier_score_loss,
    log_loss,
    precision_recall_curve # <--- Добавлено для поиска порога
)

# 1. Получаем вероятность дефолта
pd_pred = best_model.predict_proba(X_test)[:, 1]

# 2. Ищем оптимальный порог вместо фиксированных 0.5
# Вычисляем все возможные пороги и выбираем тот, который дает максимальный F1-score
precisions, recalls, thresholds = precision_recall_curve(y_test, pd_pred)

# Добавляем 1e-9 чтобы избежать деления на ноль
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

# Используем найденный порог
y_pred = (pd_pred >= optimal_threshold).astype(int)

# 3. Считаем метрики
metrics = {
    "ROC-AUC": roc_auc_score(y_test, pd_pred),
    "PR-AUC (Average Precision)": average_precision_score(y_test, pd_pred),
    "Accuracy": accuracy_score(y_test, y_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
    "Precision (default)": precision_score(y_test, y_pred, zero_division=0),
    "Recall (default)": recall_score(y_test, y_pred, zero_division=0),
    "F1 (default)": f1_score(y_test, y_pred, zero_division=0),
    "Brier Score": brier_score_loss(y_test, pd_pred),
    "Log Loss": log_loss(y_test, pd_pred),
}

print("Модель:", best_name)
print(f"Оптимальный порог классификации (по max F1): {optimal_threshold:.3f}")
display(pd.DataFrame(
    {"Metric": metrics.keys(), "Value": metrics.values()}
).round(4))

# 4. Подробный отчёт по классам
print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Нет дефолта", "Дефолт"],
    digits=3,
    zero_division=0
))

# 5. Матрица ошибок
cm = confusion_matrix(y_test, y_pred)
display(pd.DataFrame(
    cm,
    index=["Факт: нет дефолта", "Факт: дефолт"],
    columns=["Прогноз: нет дефолта", "Прогноз: дефолт"]
))

Модель: CatBoost
Оптимальный порог классификации (по max F1): 0.185


,Metric,Value
0,ROC-AUC,0.7888
1,PR-AUC (Average Precision),0.2916
2,Accuracy,0.9048
3,Balanced Accuracy,0.6604
4,Precision (default),0.3408
5,Recall (default),0.3759
6,F1 (default),0.3575
7,Brier Score,0.0584
8,Log Loss,0.2150



Classification report:
              precision    recall  f1-score   support

 Нет дефолта      0.952     0.945     0.949      3718
      Дефолт      0.341     0.376     0.358       282

    accuracy                          0.905      4000
   macro avg      0.647     0.660     0.653      4000
weighted avg      0.909     0.905     0.907      4000



,Прогноз: нет дефолта,Прогноз: дефолт
Факт: нет дефолта,3513,205
Факт: дефолт,176,106


In [24]:
import numpy as np
import pandas as pd

# Адаптируем сценарии под наш откалиброванный уровень дефолта ~7%.
# Пороги 20-40% не сработают, так как вероятности редко доходят до таких значений.
scenarios = [
    (0.03, 0.08),  # Консервативный: мало выдаем авто-одобрением, бережем портфель
    (0.05, 0.12),  # Баланс: средний риск
    (0.07, 0.15),  # Агрессивный: максимальный захват рынка
    (0.10, 0.20),  # Сверх-агрессивный
]

# Условные бизнес-метрики для оценки качества скоринга в деньгах (PnL)
AVG_LOAN = 2_000_000
MARGIN = 0.15  # Заработок банка с хорошего клиента (+15%)
LOSS = 0.50    # Потеря тела кредита при дефолте с учетом взыскания (-50%)

rows = []

for approval, rejection in scenarios:
    zones = np.select(
        [
            pd_pred < approval,
            (pd_pred >= approval) & (pd_pred < rejection)
        ],
        ["1. Одобрение", "2. Ручной разбор"],
        default="3. Отказ"
    )

    for zone in ["1. Одобрение", "2. Ручной разбор", "3. Отказ"]:
        mask = zones == zone
        if not mask.any():
            continue

        zone_y = np.asarray(y_test)[mask]
        defaults = zone_y.sum()
        goods = len(zone_y) - defaults

        # Считаем финансовый результат зоны
        if zone == "1. Одобрение":
            profit = (goods * AVG_LOAN * MARGIN) - (defaults * AVG_LOAN * LOSS)
        elif zone == "2. Ручной разбор":
            # Эмулируем работу аналитика: спасает от 50% дефолтов, но ошибочно режет 30% хороших
            profit = ((goods * 0.7) * AVG_LOAN * MARGIN) - ((defaults * 0.5) * AVG_LOAN * LOSS)
        else:
            profit = 0

        rows.append({
            "Пороги (Одобр. / Отказ)": f"<{approval:.0%} / >{rejection:.0%}",
            "Зона": zone,
            "Заявок": int(mask.sum()),
            "Доля потока, %": round(mask.mean() * 100, 1),  # Восстановлен знак умножения
            "Факт. дефолты, %": round(zone_y.mean() * 100, 1), # Восстановлен знак умножения
            "PnL (млн ₽)": round(profit / 1_000_000, 1)
        })

scenario_table = pd.DataFrame(rows)

# Красивый вывод с подсветкой убытков и сверхприбылей
display(
    scenario_table.style
    .background_gradient(subset=["PnL (млн ₽)"], cmap="RdYlGn")
    .format({
        "Доля потока, %": "{:.1f}",
        "Факт. дефолты, %": "{:.1f}",
        "PnL (млн ₽)": "{:.1f}"
    })
)

,Пороги (Одобр. / Отказ),Зона,Заявок,"Доля потока, %","Факт. дефолты, %",PnL (млн ₽)
0,<3% / >8%,1. Одобрение,1719,43.0,1.9,474.1
1,<3% / >8%,2. Ручной разбор,1339,33.5,5.1,232.9
2,<3% / >8%,3. Отказ,942,23.5,19.3,0.0
3,<5% / >12%,1. Одобрение,2443,61.1,2.9,641.9
4,<5% / >12%,2. Ручной разбор,983,24.6,6.9,158.2
5,<5% / >12%,3. Отказ,574,14.4,25.1,0.0
6,<7% / >15%,1. Одобрение,2897,72.4,3.0,754.7
7,<7% / >15%,2. Ручной разбор,685,17.1,11.1,89.9
8,<7% / >15%,3. Отказ,418,10.4,28.2,0.0
9,<10% / >20%,1. Одобрение,3280,82.0,3.8,821.5
